# 20 — Feature Engineering

**ENAI 603 Capstone — WMATA Metro Delay Prediction**

This notebook performs further feature engineering on the WMATA database with the aim of improving models performance.

**Sections:**
1. Load existing features.csv file
2. Add station has parking feature
3. Add transfer station feature
4. Add VRE, AMTRAK, MARK connection feature
5. Merge new features into existing features.csv

## 1. Load existing features.csv file

In [17]:
df_features = pd.read_csv(PROJECT_DIR / "data" / "features.csv")
df_features

,collected_at,location_name,destination_name,date,hour,day_of_week,is_weekend,is_rush_hour,line,location_code,...,lon,minutes_num,num_trains_at_station,avg_cars_at_station,delay_rate_30min,line_delay_rate_30min,active_incident,incident_is_delay,scheduled_headway_min,is_delayed
0,2026-03-11 02:00:05.155939+00:00,Metro Center,Glenmont,2026-03-10,22,1,0,0,RD,A01,...,-77.028099,1.0,6,6.666667,0.0,0.000000,0,0,0.0,0
1,2026-03-11 02:00:05.155939+00:00,Court House,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,K01,...,-77.083910,4.0,6,7.333333,0.0,0.000000,1,1,NaN,0
2,2026-03-11 02:00:05.155939+00:00,Benning Road,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G01,...,-76.938291,2.0,6,7.000000,0.0,0.000000,0,0,NaN,0
3,2026-03-11 02:00:05.155939+00:00,Addison Road-Seat Pleasant,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G03,...,-76.893592,26.0,6,7.333333,0.0,0.000000,0,0,NaN,0
4,2026-03-11 02:00:05.155939+00:00,Van Dorn Street,Franconia-Springfield,2026-03-10,22,1,0,0,BL,J02,...,-77.129407,30.0,6,6.000000,0.0,0.000000,0,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
333137,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,-77.021851,3.0,5,7.500000,0.0,0.378596,1,1,0.5,0
333138,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,-77.021851,0.0,5,7.500000,0.0,0.378161,1,1,0.5,0
333139,2026-03-18 18:30:16.166019+00:00,Farragut North,Shady Grv,2026-03-18,14,2,0,0,RD,A02,...,-77.039766,2.0,6,7.333333,0.0,0.377727,0,0,NaN,0
333140,2026-03-18 18:30:16.166019+00:00,Takoma,Glenmont,2026-03-18,14,2,0,0,RD,B07,...,-77.017834,12.0,6,7.600000,0.0,0.377294,0,0,1.5,0


---

# Feature Engineering

## 2. Add station has parking feature

In [18]:
STATION_PARKING = ['N12', 'N11', 'N09', 'N08', 'N06', 'K05', 'K08', 'K07', 'K06', 'A15', 'A14', 'A13', 'A12', 'A11', 'J03', 'J02', 'C15',
                   'F11', 'F10', 'F09', 'F08', 'F06', 'G02', 'G03', 'G04', 'G05', 'D09', 'D10', 'D11', 'D12', 'D13',
                   'B04', 'B07', 'B06', 'B08', 'B09', 'B10', 'B11', 'E07', 'E08', 'E09', 'E10']

# Check if the upcoming/current station location has parking availability
def station_parking_available(location_code):
  if location_code in STATION_PARKING:
    return 1
  else:
    return 0

# Add loc_has_parking feature based on location_code
df_features['loc_has_parking'] = df_features['location_code'].apply(station_parking_available)
df_features

,collected_at,location_name,destination_name,date,hour,day_of_week,is_weekend,is_rush_hour,line,location_code,...,minutes_num,num_trains_at_station,avg_cars_at_station,delay_rate_30min,line_delay_rate_30min,active_incident,incident_is_delay,scheduled_headway_min,is_delayed,loc_has_parking
0,2026-03-11 02:00:05.155939+00:00,Metro Center,Glenmont,2026-03-10,22,1,0,0,RD,A01,...,1.0,6,6.666667,0.0,0.000000,0,0,0.0,0,0
1,2026-03-11 02:00:05.155939+00:00,Court House,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,K01,...,4.0,6,7.333333,0.0,0.000000,1,1,NaN,0,0
2,2026-03-11 02:00:05.155939+00:00,Benning Road,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G01,...,2.0,6,7.000000,0.0,0.000000,0,0,NaN,0,0
3,2026-03-11 02:00:05.155939+00:00,Addison Road-Seat Pleasant,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G03,...,26.0,6,7.333333,0.0,0.000000,0,0,NaN,0,1
4,2026-03-11 02:00:05.155939+00:00,Van Dorn Street,Franconia-Springfield,2026-03-10,22,1,0,0,BL,J02,...,30.0,6,6.000000,0.0,0.000000,0,0,NaN,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
333137,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,3.0,5,7.500000,0.0,0.378596,1,1,0.5,0,0
333138,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,0.0,5,7.500000,0.0,0.378161,1,1,0.5,0,0
333139,2026-03-18 18:30:16.166019+00:00,Farragut North,Shady Grv,2026-03-18,14,2,0,0,RD,A02,...,2.0,6,7.333333,0.0,0.377727,0,0,NaN,0,0
333140,2026-03-18 18:30:16.166019+00:00,Takoma,Glenmont,2026-03-18,14,2,0,0,RD,B07,...,12.0,6,7.600000,0.0,0.377294,0,0,1.5,0,1


## 3. Add transfer station feature

In [19]:
TRANSFER_STATIONS = ['K05', 'C05', 'C07', 'C13', 'A01', 'C01', 'B01', 'F01', 'D03', 'F03', 'E01', 'B06', 'E06' 'D08']

# Check if the upcoming/current station location is a transfer station
def is_transfer_station(location_code):
  if location_code in TRANSFER_STATIONS:
    return 1
  else:
    return 0

# Add loc_is_transfer feature based on location_code
df_features['loc_is_transfer'] = df_features['location_code'].apply(is_transfer_station)
df_features

,collected_at,location_name,destination_name,date,hour,day_of_week,is_weekend,is_rush_hour,line,location_code,...,num_trains_at_station,avg_cars_at_station,delay_rate_30min,line_delay_rate_30min,active_incident,incident_is_delay,scheduled_headway_min,is_delayed,loc_has_parking,loc_is_transfer
0,2026-03-11 02:00:05.155939+00:00,Metro Center,Glenmont,2026-03-10,22,1,0,0,RD,A01,...,6,6.666667,0.0,0.000000,0,0,0.0,0,0,1
1,2026-03-11 02:00:05.155939+00:00,Court House,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,K01,...,6,7.333333,0.0,0.000000,1,1,NaN,0,0,0
2,2026-03-11 02:00:05.155939+00:00,Benning Road,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G01,...,6,7.000000,0.0,0.000000,0,0,NaN,0,0,0
3,2026-03-11 02:00:05.155939+00:00,Addison Road-Seat Pleasant,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G03,...,6,7.333333,0.0,0.000000,0,0,NaN,0,1,0
4,2026-03-11 02:00:05.155939+00:00,Van Dorn Street,Franconia-Springfield,2026-03-10,22,1,0,0,BL,J02,...,6,6.000000,0.0,0.000000,0,0,NaN,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
333137,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,5,7.500000,0.0,0.378596,1,1,0.5,0,0,1
333138,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,5,7.500000,0.0,0.378161,1,1,0.5,0,0,1
333139,2026-03-18 18:30:16.166019+00:00,Farragut North,Shady Grv,2026-03-18,14,2,0,0,RD,A02,...,6,7.333333,0.0,0.377727,0,0,NaN,0,0,0
333140,2026-03-18 18:30:16.166019+00:00,Takoma,Glenmont,2026-03-18,14,2,0,0,RD,B07,...,6,7.600000,0.0,0.377294,0,0,1.5,0,1,0


## 4. Add VRE, AMTRAK, MARK connection feature

In [20]:
RAIL_SYS_CONNECT = {
  'VRE': ['B03', 'C13', 'C09', 'J03', 'D03', 'F03'],
  'AMTRAK': ['D13', 'B03', 'C13', 'A14'],
  'MARK':  ['D13', 'E10', 'E09', 'B03', 'B08', 'A14']
}

# Check if the upcoming/current station location connects to any other rail systems
def connect_rail_systems(location_code):
  connects = []
  
  for rail, stations in RAIL_SYS_CONNECT.items():
    if location_code in stations:
      connects.append(1)
    else:
      connects.append(0)
  
  return connects

# Add VRE, AMTRAK, MARK connection feature based on location_code
df_features[['loc_conn_vre', 'loc_conn_amtrak', 'loc_conn_mark']] = df_features['location_code'].apply(connect_rail_systems).to_list()
df_features

,collected_at,location_name,destination_name,date,hour,day_of_week,is_weekend,is_rush_hour,line,location_code,...,line_delay_rate_30min,active_incident,incident_is_delay,scheduled_headway_min,is_delayed,loc_has_parking,loc_is_transfer,loc_conn_vre,loc_conn_amtrak,loc_conn_mark
0,2026-03-11 02:00:05.155939+00:00,Metro Center,Glenmont,2026-03-10,22,1,0,0,RD,A01,...,0.000000,0,0,0.0,0,0,1,0,0,0
1,2026-03-11 02:00:05.155939+00:00,Court House,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,K01,...,0.000000,1,1,NaN,0,0,0,0,0,0
2,2026-03-11 02:00:05.155939+00:00,Benning Road,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G01,...,0.000000,0,0,NaN,0,0,0,0,0,0
3,2026-03-11 02:00:05.155939+00:00,Addison Road-Seat Pleasant,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G03,...,0.000000,0,0,NaN,0,1,0,0,0,0
4,2026-03-11 02:00:05.155939+00:00,Van Dorn Street,Franconia-Springfield,2026-03-10,22,1,0,0,BL,J02,...,0.000000,0,0,NaN,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
333137,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,0.378596,1,1,0.5,0,0,1,0,0,0
333138,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,0.378161,1,1,0.5,0,0,1,0,0,0
333139,2026-03-18 18:30:16.166019+00:00,Farragut North,Shady Grv,2026-03-18,14,2,0,0,RD,A02,...,0.377727,0,0,NaN,0,0,0,0,0,0
333140,2026-03-18 18:30:16.166019+00:00,Takoma,Glenmont,2026-03-18,14,2,0,0,RD,B07,...,0.377294,0,0,1.5,0,1,0,0,0,0


## 5. Save new features into existing features.csv

In [26]:
df_features.to_csv(PROJECT_DIR / "data" / 'features.csv')